In [1]:
"""LS Open API t1101 — 국내 주식 현재가·종목명(hname) 등."""

import json
import logging
from json.decoder import JSONDecodeError

import requests
from requests.exceptions import RequestException

from ls_auth import api_manager, get_token_futures

logger = logging.getLogger(__name__)

TR = "t1101"
BASE_URL = "https://openapi.ls-sec.co.kr:8080"
PATH = "stock/market-data"
URL = f"{BASE_URL}/{PATH}"
# KRX 초당 약 3회 제한 (simple_trading t2101 등과 동일 간격)
T1101_MIN_INTERVAL_SEC = 0.34


def get_current(shcode="005930"):
    """
    t1101OutBlock dict 반환. 실패 시 None.
    shcode: 6자리 종목코드 문자열.
    """
    shcode = str(shcode).strip().zfill(6)
    access_token = get_token_futures()
    header = {
        "content-type": "application/json; charset=utf-8",
        "Authorization": f"Bearer {access_token}",
        "tr_cd": TR,
        "tr_cont": "N",
        "tr_cont_key": "",
    }
    body = {f"{TR}InBlock": {"shcode": shcode}}

    api_manager.wait_for_next_call(TR, T1101_MIN_INTERVAL_SEC)
    try:
        res = requests.post(URL, headers=header, data=json.dumps(body), timeout=30)
        res.raise_for_status()
        res_json = res.json()
        if f"{TR}OutBlock" not in res_json:
            logger.error("t1101: OutBlock 없음: %s", res_json)
            return None
        return res_json[f"{TR}OutBlock"]
    except (RequestException, JSONDecodeError, KeyError) as e:
        logger.error("t1101 호출 오류: %s", e)
        return None


In [2]:
get_current("005930")

{'hname': '삼성전자',
 'price': 219000,
 'sign': '5',
 'change': 3000,
 'diff': '-1.35',
 'volume': 2411707,
 'jnilclose': 222000,
 'offerho1': 219000,
 'bidho1': 218500,
 'offerrem1': 35411,
 'bidrem1': 89215,
 'preoffercha1': 0,
 'prebidcha1': 0,
 'offerho2': 219500,
 'bidho2': 218000,
 'offerrem2': 41835,
 'bidrem2': 113021,
 'preoffercha2': 0,
 'prebidcha2': 0,
 'offerho3': 220000,
 'bidho3': 217500,
 'offerrem3': 54160,
 'bidrem3': 114982,
 'preoffercha3': 0,
 'prebidcha3': 0,
 'offerho4': 220500,
 'bidho4': 217000,
 'offerrem4': 61521,
 'bidrem4': 201099,
 'preoffercha4': 0,
 'prebidcha4': 0,
 'offerho5': 221000,
 'bidho5': 216500,
 'offerrem5': 44140,
 'bidrem5': 89331,
 'preoffercha5': 0,
 'prebidcha5': 0,
 'offerho6': 221500,
 'bidho6': 216000,
 'offerrem6': 45476,
 'bidrem6': 93229,
 'preoffercha6': 0,
 'prebidcha6': 0,
 'offerho7': 222000,
 'bidho7': 215500,
 'offerrem7': 77846,
 'bidrem7': 183427,
 'preoffercha7': 0,
 'prebidcha7': 0,
 'offerho8': 222500,
 'bidho8': 215000,
 'o